In [0]:
CREATE TABLE IF NOT EXISTS sac.network.log (
		timestamp TIMESTAMP,
		customer_id STRING,
		issue_detected STRING,
		speed_measured_mbps DOUBLE,
		packet_loss_percent DOUBLE,
		latency_ms DOUBLE,
		downtime_minutes INTEGER,
		connection_drops_count INTEGER,
		CONSTRAINT log_pk PRIMARY KEY (timestamp),
		CONSTRAINT log_fk FOREIGN KEY (customer_id) REFERENCES sac.customer.customer (customer_id)
	);

-- Edit constraints of table
ALTER TABLE
	sac.network.log
DROP CONSTRAINT IF EXISTS
	log_connection_ts;

ALTER TABLE
	sac.network.log
DROP CONSTRAINT IF EXISTS
	log_connection_sm;

ALTER TABLE
	sac.network.log
DROP CONSTRAINT IF EXISTS
	log_connection_pl;

ALTER TABLE
	sac.network.log
DROP CONSTRAINT IF EXISTS
	log_connection_l;

ALTER TABLE
	sac.network.log
DROP CONSTRAINT IF EXISTS
	log_connection_dt;

ALTER TABLE
	sac.network.log
DROP CONSTRAINT IF EXISTS
	log_connection_cd;

ALTER TABLE
	sac.network.log
ADD
	CONSTRAINT log_connection_ts CHECK (timestamp <= current_date());

ALTER TABLE
	sac.network.log
ADD
	CONSTRAINT log_connection_sm CHECK (speed_measured_mbps >= 0);

ALTER TABLE
	sac.network.log
ADD
	CONSTRAINT log_connection_pl
		CHECK (
			packet_loss_percent >= 0
			AND packet_loss_percent <= 100
		);

ALTER TABLE
	sac.network.log
ADD
	CONSTRAINT log_connection_l CHECK (latency_ms >= 0);

ALTER TABLE
	sac.network.log
ADD
	CONSTRAINT log_connection_dt CHECK (downtime_minutes >= 0);

ALTER TABLE
	sac.network.log
ADD
	CONSTRAINT log_connection_cd CHECK (connection_drops_count >= 0);

-- Fill table with values
MERGE INTO
	sac.network.log s
USING (
	SELECT
		CAST(l.timestamp AS TIMESTAMP) AS timestamp,
		l.customer_id,
		l.issue_detected,
		CAST(l.speed_measured_mbps AS DOUBLE) AS speed_measured_mbps,
		CAST(l.packet_loss_percent AS DOUBLE) AS packet_loss_percent,
		CAST(l.latency_ms AS DOUBLE) AS latency_ms,
		CAST(l.downtime_minutes AS INTEGER) AS downtime_minutes,
		CAST(l.connection_drops_count AS INTEGER) AS connection_drops_count
	FROM
		sac.network.log_bronze l
	QUALIFY
		row_number() OVER (PARTITION BY l.timestamp ORDER BY l.ingestion_time DESC) = 1
) b
ON
	s.timestamp = b.timestamp
WHEN NOT MATCHED THEN INSERT *;